# 12b · More training data, on Stockfish's moves (`sft_100k_stockfish`)

**The question.** Notebook 08 found that training SFT on Stockfish's best moves instead of the players' moves made no significant difference with 20,000 positions (`sft_stockfish` 7.3% vs. `sft_v1` 7.8%, p = 0.6). One explanation is that 20,000 examples are too few for the model to learn *which* move is best from either kind of answer. This notebook asks: **with 100,000 positions, do Stockfish's moves help?**

It is notebook 12 with one change, the answer the model learns, so together the four SFT models form a 2 × 2 comparison:

| | 20,000 positions | 100,000 positions |
|---|---|---|
| **Player's move** | `sft_v1` (notebook 03) | `sft_100k` (notebook 12) |
| **Stockfish's move** | `sft_stockfish` (notebook 08) | **`sft_100k_stockfish` (this notebook)** |

**What changes, and what stays the same.**

| | `sft_100k` (notebook 12) | `sft_100k_stockfish` (this notebook) |
|---|---|---|
| Training positions | 100,000 (the first 20,000 are `sft_v1`'s) | the same 100,000, in the same order |
| Answer the model learns | the move the player made | **Stockfish's best move** (search depth 10, as in notebook 08) |
| Model, LoRA, learning rate, epochs (1), seed, prompt, copies after 20k/40k/60k/80k | | the same |
| Graded on | 1,000 test puzzles | the same 1,000 puzzles, same grader |

As in notebook 12, one run gives a whole curve: copies of the model are saved after 20k, 40k, 60k and 80k examples and graded. The same caveat applies: an intermediate copy has not finished the learning-rate schedule. Comparing the 20k copy with `sft_stockfish` (the same 20,000 positions and answers) shows how much this matters.

**Labelling.** Stockfish's best move has to be computed for each training position. The first 20,000 positions are exactly notebook 08's, so their labels are reused from `data/processed/sft_stockfish_train.parquet`; only the other 80,000 are new. They are labelled in chunks of 10,000, each saved to `data/processed/sft_100k_stockfish_labels/` as soon as it is done, so a stopped Job keeps its work and a rerun continues where it stopped.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Plain *Serverless* has no GPU. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run. (The first cell restarts Python once; if Run all stops there, click Run all again.)
3. Set `QUICK_CHECK = False` and run the real thing as a **Job**, so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification. Separate Jobs can run at the same time.
4. Afterwards, commit and push **only `results/`** from the Git folder. Leave `notebooks/` unticked: Databricks cannot save notebook outputs to git, so committing a notebook from there would erase its outputs on GitHub. The outputs of a Job run are on the Job's run page; every figure and table is also saved in `results/figures/`.

**Time:** about 1–1.5 hours of labelling and 5 hours of training on an A10, plus about 20 minutes of grading (roughly $12–15 of credits).

**Outputs**
- the labels in `data/processed/sft_100k_stockfish_labels/` (not on GitHub)
- the adapter in `checkpoints/sft_100k_stockfish/` and the copies in `checkpoints/sft_100k_stockfish_at_20k/`, `_at_40k/`, `_at_60k/`, `_at_80k/` (not on GitHub)
- results: `results/sft_100k_stockfish_training.json`, and for the final model and each copy `results/<name>.json`, `results/<name>_dev.json`, and rows in `results/summary.csv`
- every figure and table of this experiment in `results/figures/sft_100k_stockfish/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
RUN_NAME = "sft_100k_stockfish"
PLAYED_RUN = "sft_100k"                  # notebook 12: the same positions, with the players' moves
QUICK_CHECK = False   # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
TRAIN_EXAMPLES = 100_000                 # as in notebook 12
SAVE_AT = [20_000, 40_000, 60_000, 80_000]  # save and grade a copy after this many examples
LABEL_WORKERS = 4     # Stockfish processes running at once while labelling (as in notebook 08)
CHUNK = 10_000        # positions labelled and saved at a time

import gc
import time

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.data import add_stockfish_moves
from chesslm.evaluate import load_eval_model
from chesslm.experiments import compare_with, figures_dir, grade_puzzles, puzzle_table, save_table, save_training
from chesslm.train_sft import copy_dir, examples_label, sample_examples, to_dataset, train

sft_cfg = load_config("sft")
sft_cfg["data"]["answer"] = "stockfish"   # THE change compared with notebook 12; everything else is the same
prompt_cfg, eval_cfg = load_config("prompt"), load_config("eval")
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
SEED = sft_cfg["training"]["seed"]
SFT_V1_EXAMPLES = sft_cfg["data"]["train_examples"]          # 20,000
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
sft_cfg["data"]["train_examples"] = TRAIN_EXAMPLES
sft_cfg["training"]["eval_steps"] = 500                      # validation loss less often, to save time (as in 12)

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUN_NAME += "_check"
    sft_cfg["data"].update(trial_examples=16, train_examples=96, val_examples=16)
    sft_cfg["training"].update(logging_steps=1, eval_steps=1)
    SAVE_AT = [32, 64]
    CHUNK = 40
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
ADAPTER_DIR = REPO_ROOT / "checkpoints" / RUN_NAME
FIGURES = figures_dir(RUN_NAME)
print(f"Run: {RUN_NAME} | answer learned: {sft_cfg['data']['answer']} | {sft_cfg['data']['train_examples']:,} "
      f"training examples | copies after {SAVE_AT} | figures and tables -> {FIGURES}")

## 1. Training data: the same 100,000 positions as notebook 12, labelled by Stockfish

The same seeded sample as notebook 12 (the cell checks that its first 20,000 positions are `sft_v1`'s). Labels from notebook 08 are reused for the first 20,000 training positions and the 1,000 validation positions; the rest are labelled here, one chunk at a time. If a chunk was already labelled by an earlier run of this notebook, it is loaded instead.

In [ ]:
n = sft_cfg["data"]
PROCESSED = REPO_ROOT / "data" / "processed"
CHUNKS = PROCESSED / f"{RUN_NAME}_labels"
label_cfg = {**eval_cfg["stockfish"], "depth": n["stockfish_depth"]}

train_rows = sample_examples(train_positions, n["train_examples"], SEED)
same_start = list(train_rows.fen.iloc[:SFT_V1_EXAMPLES]) == list(sample_examples(train_positions, SFT_V1_EXAMPLES, SEED).fen)
assert same_start or QUICK_CHECK, "the first 20,000 positions should be sft_v1's"
val_rows = sample_examples(pd.read_parquet(PROCESSED / "val.parquet"), n["val_examples"], SEED)


def label(sample, split):
    """Stockfish's best move for every position in sample (columns best_uci, best_san), in the same order."""
    parts = []
    from_08 = PROCESSED / f"sft_stockfish_{split}.parquet"   # notebook 08's labels for its first positions
    if from_08.exists() and not QUICK_CHECK:
        done = pd.read_parquet(from_08)
        assert list(done.fen) == list(sample.fen.iloc[: len(done)]), f"{from_08.name} holds other positions"
        parts.append(done)
    start = sum(len(p) for p in parts)
    print(f"{split}: {start:,} positions labelled in notebook 08, {len(sample) - start:,} to label here")
    for i, first in enumerate(range(start, len(sample), CHUNK)):
        chunk = sample.iloc[first:first + CHUNK]
        path = CHUNKS / f"{split}_{i:02d}.parquet"
        if path.exists() and not QUICK_CHECK:
            saved = pd.read_parquet(path)
            assert list(saved.fen) == list(chunk.fen), f"{path} holds other positions; delete it and rerun"
            parts.append(saved)
            print(f"  positions {first:,}-{first + len(chunk):,}: loaded from {path.name}")
            continue
        begin = time.time()
        labelled = add_stockfish_moves(chunk, label_cfg, LABEL_WORKERS)
        if not QUICK_CHECK:
            CHUNKS.mkdir(exist_ok=True)
            labelled.to_parquet(path)
        parts.append(labelled)
        print(f"  positions {first:,}-{first + len(chunk):,}: labelled in {(time.time() - begin) / 60:.1f} min")
    result = pd.concat(parts)
    assert list(result.fen) == list(sample.fen)
    return result


train_rows = label(train_rows, "train")
val_rows = label(val_rows, "val")

same = train_rows.move_uci == train_rows.best_uci
data_table = pd.DataFrame({
    "Training positions": [SFT_V1_EXAMPLES, len(train_rows)],
    "Player's move = Stockfish's move (%)": [round(same.iloc[:SFT_V1_EXAMPLES].mean() * 100, 1), round(same.mean() * 100, 1)],
}, index=pd.Index([f"first {SFT_V1_EXAMPLES:,} (sft_stockfish's data)", f"all {len(train_rows):,} (this run)"],
                  name="Training data"))
save_table(data_table, FIGURES, "training_data",
           "How often the move the 2000+ player made equals Stockfish's best move (depth 10). Where they differ, "
           "this run learns a different answer than sft_100k.")
display(data_table)

## 2. Trial run

A short run on the first few hundred training examples, to check that the loss goes down on this machine before the long run.

In [ ]:
trial_ds = to_dataset(train_rows.head(n["trial_examples"]), prompt_cfg, answer="stockfish")
val_ds = to_dataset(val_rows, prompt_cfg, answer="stockfish")
example = trial_ds[0]
print("Prompt:    ", example["prompt"][0]["content"].replace("\n", "\n            "))
print("Completion:", example["completion"][0]["content"], "(Stockfish's move)\n")
trial_log, trial_compute = train(trial_ds, val_ds.select(range(min(100, len(val_ds)))), sft_cfg,
                                 REPO_ROOT / "checkpoints" / f"{RUN_NAME}_trial")
losses = trial_log.loss.dropna()
print(f"Training loss: {losses.iloc[0]:.3f} at the start -> {losses.iloc[-1]:.3f} at the end")
gc.collect()
torch.cuda.empty_cache()

## 3. Full training run, saving copies along the way

The figure shows the loss on training examples and on the 1,000 held-out validation positions (also labelled by Stockfish).

In [ ]:
train_ds = to_dataset(train_rows, prompt_cfg, answer="stockfish")
print(f"train: {len(train_ds):,} examples  validation: {len(val_ds):,} examples")
log, compute = train(train_ds, val_ds, sft_cfg, ADAPTER_DIR, save_at=SAVE_AT)
save_training(RUN_NAME, {"sft": sft_cfg, "prompt": prompt_cfg, "save_at": SAVE_AT}, compute, log)
plots.loss_curve(log, FIGURES / "training_loss.png", title=f"{RUN_NAME}: loss during training");

## 4. Grade every copy and the final model on the puzzles

Exactly the grader, puzzles, and greedy decoding used for every other model: the 200 dev puzzles first, then the 1,000 test puzzles. Copies that are missing (for example, if training stopped early) are skipped.

In [ ]:
models = {f"{RUN_NAME}_at_{examples_label(k)}": (k, copy_dir(ADAPTER_DIR, k)) for k in SAVE_AT}
models[RUN_NAME] = (n["train_examples"], ADAPTER_DIR)
for name, (examples, adapter) in models.items():
    if not adapter.exists():
        print(f"{name}: skipped, {adapter} not found")
        continue
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter)
    grade_puzzles(model, tokenizer, "dev", f"{name}_dev", configs, str(adapter), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", name, configs, str(adapter), PUZZLE_LIMIT)
    del model

## 5. Results: do Stockfish's moves help with more data?

**How to read this section**
- **The 2 × 2 table** gives the puzzle accuracy of the four SFT models. Compare the rows (player's move vs. Stockfish's move) at 20,000 and at 100,000 positions: if Stockfish's moves only help with enough data, the gap between the rows opens up in the 100,000 column.
- **The scaling curves** show puzzle accuracy and the share of legal answers against the number of training examples seen, for this run and for notebook 12's run on the players' moves, on the same axes. A dashed line marks a random legal move (5.6%). The curve of `sft_100k` appears once notebook 12 has finished.
- **The paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance. `sft_100k_stockfish` vs. `sft_100k` is the effect of the answer source with 100,000 positions; vs. `sft_stockfish`, the effect of the extra data with Stockfish's moves.

All numbers are on the 1,000 test puzzles.

In [ ]:
graded = set(pd.read_csv(experiments.RESULTS / "summary.csv").run)


def scaling_table(run):
    """Test-puzzle results of a run's saved copies and final model, indexed by training examples seen (None if not graded)."""
    names = {f"{run}_at_{examples_label(k)}": k for k in SAVE_AT} | {run: n["train_examples"]}
    names = {name: k for name, k in names.items() if name in graded}
    if not names:
        return None
    table = puzzle_table(list(names)).reset_index()
    table.insert(1, "Training examples", list(names.values()))
    return table.set_index("Training examples")[["Model", "Puzzle accuracy (%)", "Legal (%)", "ACPL (cp)"]]


scaling = scaling_table(RUN_NAME)
save_table(scaling, FIGURES, "scaling",
           "Test-puzzle results of the copies saved during training and of the final model. ACPL is over legal answers only.")
display(scaling)

accuracy = lambda run: puzzle_table([run]).iloc[0]["Puzzle accuracy (%)"] if run in graded else None
two_by_two = pd.DataFrame({"20,000 positions": [accuracy("sft_v1"), accuracy("sft_stockfish")],
                           "100,000 positions": [accuracy(PLAYED_RUN), accuracy(RUN_NAME)]},
                          index=pd.Index(["Player's move", "Stockfish's move"], name="Answer learned"))
save_table(two_by_two, FIGURES, "two_by_two",
           "Test-puzzle accuracy (%) of the four SFT models: sft_v1, sft_100k (top row); sft_stockfish, "
           "sft_100k_stockfish (bottom row). Empty = not graded yet.")
display(two_by_two)

panels = [("Puzzle accuracy (%)", "puzzle accuracy (%)"), ("Legal (%)", "legal answers (%)")]
curves = {f"Stockfish's move ({RUN_NAME})": scaling}
played = scaling_table(PLAYED_RUN)
if played is not None:
    curves = {f"player's move ({PLAYED_RUN})": played, **curves}
else:
    print(f"{PLAYED_RUN} is not graded yet, so its curve is left out; rerun this cell once notebook 12 has finished.")
plots.scaling_curves(curves, panels, {"random legal move": {"Puzzle accuracy (%)": 5.6}},
                     FIGURES / "scaling_curves.png", title="Does SFT improve with more data, and does the answer source matter?");

In [ ]:
runs = [run for run in ["sft_v1", "sft_stockfish", PLAYED_RUN, RUN_NAME] if run in graded]
display(puzzle_table(runs, FIGURES))

pairs = [(RUN_NAME, other) for other in [PLAYED_RUN, "sft_stockfish", "sft_v1"] if other in graded]
first_copy = f"{RUN_NAME}_at_{examples_label(SAVE_AT[0])}"
if first_copy in graded:
    pairs.append((first_copy, "sft_stockfish"))
comparison = pd.DataFrame([compare_with(new, old) for new, old in pairs]).set_index("model")
save_table(comparison, FIGURES, "paired_comparison",
           "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
display(comparison)

In [ ]:
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
plots.compare_metrics(summary, runs, FIGURES / "puzzle_metrics.png",
                      title="The SFT models: players' vs. Stockfish's moves, 20,000 vs. 100,000 positions");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png",
                       title="Puzzle accuracy by puzzle difficulty (rating band)");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png", title="Are the answers legal moves?");

## Summary of outputs

- `results/figures/sft_100k_stockfish/`: `training_data` (table), `training_loss.png`, `scaling` (table), `two_by_two` (table), `scaling_curves.png`, `puzzle_results` (table), `paired_comparison` (table), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/sft_100k_stockfish.json` and `results/sft_100k_stockfish_at_<N>k.json`: every test answer and its grade; `results/sft_100k_stockfish_training.json`: settings, compute, training log
- The final model is also graded on held-out game positions and in full games in notebook 11.
- The best of the four SFT models is the natural starting model for the combinations in notebook 15.